# 04. Modeling
The goal of this notebook is to train and compare supervised models that estimate customer churn probability.

This stage focuses on selecting an appropriate modeling approach for risk ranking, rather than producing a final decision rule. Multiple model classes are evaluated to assess how well they combine weak individual signals into a meaningful churn risk score.

The output of this stage is a candidate predictive model (or small set of models) to be used in the subsequent evaluation of retention strategies.

## 4.1. Modeling Framing

Task: binary classification with probabilistic output.

Model output: churn probability, used for customer risk ranking and downstream threshold-based actions.

Multiple model classes are evaluated (baseline, interpretable, flexible) to assess trade-offs between ranking performance, stability, and interpretability.

## 4.2. Load artifacts from Data Preparation

In [30]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss )

In [31]:
м=X_train = pd.read_csv("../data/X_train.csv")
X_test  = pd.read_csv("../data/X_test.csv")

y_train = pd.read_csv("../data/y_train.csv")["churn"]
y_test  = pd.read_csv("../data/y_test.csv")["churn"]

### 4.2.1. Sanity checks + minimal cleaning for baseline

In [32]:
print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(4800, 17) (1200, 17)
0.33791666666666664 0.3383333333333333


In [33]:
assert list(X_train.columns) == list(X_test.columns), "Train/test feature columns mismatch."
assert y_train.isin([0, 1]).all() and y_test.isin([0, 1]).all(), "y must be binary (0/1)."

### 4.2.2. Convert common boolean string values to 0/1 (if present)

In [34]:
obj_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
if obj_cols:
    for c in obj_cols:
        X_train[c] = X_train[c].replace({"True": 1, "False": 0, "true": 1, "false": 0})
        X_test[c]  = X_test[c].replace({"True": 1, "False": 0, "true": 1, "false": 0})

### 4.2.3. Re-check object columns; if any remain, they are likely categorical -> baseline needs encoding

In [35]:
obj_cols_after = X_train.select_dtypes(include=["object"]).columns.tolist()

if obj_cols_after:
    print("Categorical columns detected:", obj_cols_after)

Categorical columns detected: ['plan', 'region', 'industry']


## 4.3 Baseline Model
Establish a simple probabilistic baseline for churn risk ranking. Logistic Regression with standardized features is used as a reference model for subsequent comparisons.

In [36]:
# feature groups
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
num_cols = X_train.select_dtypes(exclude=["object"]).columns.tolist()

# pipelines
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore")),
])

# preprocessing
preprocess_lr = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ],
    remainder="drop"
)

# baseline model
baseline_lr = Pipeline([
    ("preprocess", preprocess_lr),
    ("model", LogisticRegression(max_iter=2000))
])

baseline_lr.fit(X_train, y_train)

p_train = baseline_lr.predict_proba(X_train)[:, 1]
p_test  = baseline_lr.predict_proba(X_test)[:, 1]

baseline_results = pd.DataFrame({
    "split": ["train", "test"],
    "auc_roc": [roc_auc_score(y_train, p_train), roc_auc_score(y_test, p_test)],
    "avg_precision": [average_precision_score(y_train, p_train), average_precision_score(y_test, p_test)],
    "log_loss": [log_loss(y_train, p_train), log_loss(y_test, p_test)],
    "brier_score": [brier_score_loss(y_train, p_train), brier_score_loss(y_test, p_test)],
})

baseline_results

,split,auc_roc,avg_precision,log_loss,brier_score
0,train,0.719108,0.558207,0.570121,0.194268
1,test,0.749693,0.596967,0.550285,0.185601
